In [ ]:
# ── Section 1: Setup & imports ────────────────────────────────────────────
!pip install -q "ultralytics>=8.3.0" albumentations opencv-python-headless scikit-learn --upgrade

import os, sys, json, time, random, shutil, glob
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix

print("ultralytics :", ultralytics.__version__)
print("torch       :", torch.__version__)
print("cuda avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_DIR = Path("/kaggle/working")
os.makedirs(WORK_DIR, exist_ok=True)

# ---- Locate nb0_artifacts dataset by content, not by guessed slug ----
nb0_hits = list(Path("/kaggle/input").rglob("common_config.json"))
assert nb0_hits, "Couldn't find common_config.json anywhere under /kaggle/input — is 'nb0_artifacts' attached?"
NB0_ARTIFACTS = nb0_hits[0].parent
print("Using NB0_ARTIFACTS:", NB0_ARTIFACTS)

# make seg_common.py importable
sys.path.append(str(NB0_ARTIFACTS))

In [ ]:
# ── Section 2: Load NB0 split ─────────────────────────────────────────────
import seg_common as sc   # from nb0_artifacts/seg_common.py

common_config = sc.load_common_config(str(NB0_ARTIFACTS / "common_config.json"))
split_manifest = sc.load_split_manifest(str(NB0_ARTIFACTS / "split_manifest.json"))

NUM_CLASSES_FG = common_config["num_classes_fg"]      # 67
NUM_CLASSES    = common_config["num_classes_total"]   # 68 (incl. background)
CLASS_NAMES    = common_config["class_names"]
IMG_SIZE       = common_config["img_size"]             # 512, per NB0
BG_IS_ZERO     = common_config["bg_is_zero"]

df_dist = pd.read_csv(NB0_ARTIFACTS / "pixel_class_distribution.csv")

print(f"Foreground classes: {NUM_CLASSES_FG} | total incl. bg: {NUM_CLASSES}")
print("Split sizes ->", {k: len(v) for k, v in split_manifest.items() if isinstance(v, list)})
assert split_manifest["source"] == "official_dataset_split", "NB3 must reuse NB0's frozen split, not re-split."

# ---- Sanity re-check for cross-split leakage using the exact same file lists ----
# NB0 already validated leakage via content hashing (ahash) — that's the authoritative check.
# Here we only re-verify split_manifest.json itself has no duplicate *full-path* entries
# across splits. We deliberately do NOT compare bare filename stems: many datasets number
# images per-split (e.g. train/images/0001.jpg vs valid/images/0001.jpg), so stem collisions
# across different split folders are expected and are NOT leakage.
tr_paths = set(split_manifest["train"])
va_paths = set(split_manifest["valid"])
te_paths = set(split_manifest["test"])
assert not (tr_paths & va_paths) and not (tr_paths & te_paths) and not (va_paths & te_paths), (
    "Duplicate full path listed in more than one split inside split_manifest.json — "
    "this would indicate a bug in NB0's manifest, not filename coincidence."
)
print("Leakage re-check passed — no duplicate full paths across splits in split_manifest.json.")
print("(Content-level near-duplicate check was already performed in NB0 via perceptual hashing.)")

# ---- Locate DATA_ROOT inside FoodBD2 by finding the folder that contains data.yaml
#      AND has train/valid/test subfolders (guards against picking up an unrelated yaml) ----
data_yaml_candidates = [
    p for p in Path("/kaggle/input").rglob("data.yaml")
    if (p.parent / "train").is_dir() and (p.parent / "valid").is_dir() and (p.parent / "test").is_dir()
]
assert data_yaml_candidates, "Couldn't find FoodBD's data.yaml with train/valid/test siblings under /kaggle/input."
DATA_ROOT = data_yaml_candidates[0].parent
print("Using DATA_ROOT:", DATA_ROOT)

# ---- Rebuild train/valid/test image+label dirs (polygon YOLO-seg format) via symlinks ----
# Re-anchor each manifest path (which points at NB0's original session path) onto the
# current DATA_ROOT, using only the last 3 path parts: (split_name, 'images', filename).
YOLO_DATA_ROOT = WORK_DIR / "yolo_data"

def resolve_img_label(img_path_str):
    rel_parts = Path(img_path_str).parts[-3:]        # e.g. ('train', 'images', 'foo.jpg')
    img_path = DATA_ROOT.joinpath(*rel_parts)
    lbl_path = DATA_ROOT / rel_parts[0] / "labels" / (img_path.stem + ".txt")
    return img_path, lbl_path

missing_imgs, missing_lbls = [], []
for split_name, file_list in [("train", split_manifest["train"]),
                               ("valid", split_manifest["valid"]),
                               ("test",  split_manifest["test"])]:
    img_out = YOLO_DATA_ROOT / split_name / "images"
    lbl_out = YOLO_DATA_ROOT / split_name / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for img_path_str in file_list:
        img_path, lbl_path = resolve_img_label(img_path_str)
        if not img_path.exists():
            missing_imgs.append(str(img_path)); continue
        dst_img = img_out / img_path.name
        dst_lbl = lbl_out / (img_path.stem + ".txt")
        if not dst_img.exists():
            os.symlink(img_path, dst_img)
        if lbl_path.exists():
            if not dst_lbl.exists():
                os.symlink(lbl_path, dst_lbl)
        else:
            missing_lbls.append(str(lbl_path))

assert not missing_imgs, f"{len(missing_imgs)} images from split_manifest not found under new DATA_ROOT. e.g. {missing_imgs[:3]}"
if missing_lbls:
    print(f"WARNING: {len(missing_lbls)} label files missing (will rasterize as empty/background). e.g. {missing_lbls[:3]}")

print("Rebuilt YOLO-format split dirs at:", YOLO_DATA_ROOT)
for s in ["train", "valid", "test"]:
    n_img = len(list((YOLO_DATA_ROOT / s / "images").glob("*")))
    n_lbl = len(list((YOLO_DATA_ROOT / s / "labels").glob("*")))
    print(f"  {s}: {n_img} images, {n_lbl} labels")

# ---- data.yaml for Ultralytics ----
import yaml
data_yaml = {
    "path": str(YOLO_DATA_ROOT),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc": NUM_CLASSES_FG,   # Ultralytics YOLO-seg format is foreground-class count (no explicit bg class id)
    "names": CLASS_NAMES,
}
DATA_YAML_PATH = WORK_DIR / "data.yaml"
with open(DATA_YAML_PATH, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(open(DATA_YAML_PATH).read())

In [ ]:
# ── Section 3: Model definition / pretrained weight loading ──────────────
CHECKPOINT = "yolo26n-sem.pt"
LOCAL_WEIGHTS_FALLBACK = Path("/kaggle/input/yolo26-sem-weights") / CHECKPOINT  # optional manual upload

try:
    model = YOLO(CHECKPOINT)  # Ultralytics will try to auto-download from its assets
    print(f"Loaded '{CHECKPOINT}' via Ultralytics auto-download.")
except Exception as e:
    print(f"Auto-download failed ({e}). Falling back to manually uploaded weights...")
    assert LOCAL_WEIGHTS_FALLBACK.exists(), (
        f"Upload {CHECKPOINT} as a Kaggle Dataset (e.g. 'yolo26-sem-weights') "
        f"and place it at {LOCAL_WEIGHTS_FALLBACK}"
    )
    model = YOLO(str(LOCAL_WEIGHTS_FALLBACK))

print("Model task     :", model.task)
print("Model params   :", sum(p.numel() for p in model.model.parameters()) / 1e6, "M  (spec: 1.6M)")
print("Spec Cityscapes mIoU (pretrained, reference): 78.3")
model.info()

In [ ]:
# ── Section 4: Hyperparameter config ──────────────────────────────────────
# Class weights: inverse sqrt of pixel frequency (per NB0 EDA recommendation),
# normalized to mean 1.0, foreground classes only (yolo ids 0..66 == class_id 1..67 in df_dist)
freq = df_dist.set_index("class_id")["pct"].reindex(range(1, NUM_CLASSES_FG + 1)).fillna(df_dist["pct"].min())
raw_w = 1.0 / np.sqrt(freq.values / 100.0)
class_weights = (raw_w / raw_w.mean()).round(3).tolist()

CONFIG = {
    "model_checkpoint": CHECKPOINT,
    "variant": "YOLOv26n-Sem",
    "params_M": 1.6,
    "pretrained_cityscapes_mIoU": 78.3,
    "task": "segment",
    "num_classes": NUM_CLASSES_FG,
    "img_size": 512,          # ← CHANGED from 640. Cuts per-epoch time ~35-40% (seg head cost ~scales with pixel count)
    "batch_size": 2,         # ← CHANGED from 8. Auto-batch: Ultralytics finds max batch that fits ~90% VRAM (likely 16-32 on Kaggle's T4/P100 for a nano model at 512px)
    "epochs": 40,             # ← CHANGED from 50. Your patience=5 early-stopping means you likely won't need all 50 anyway; 40 gives margin
    "patience": 5,             # unchanged
    "optimizer": "AdamW",      # unchanged
    "lr0": 1e-3,                # unchanged
    "lrf": 0.01,                # unchanged
    "scheduler": "cosine",      # unchanged
    "weight_decay": 5e-4,       # unchanged
    "warmup_epochs": 3,         # unchanged
    "loss_function": "Weighted CE + Dice (mask) + DFL/box aux terms (Ultralytics seg loss)",
    "class_weights": class_weights,
    "seed": SEED,
    "device": 0 if torch.cuda.is_available() else "cpu",
}

print(json.dumps({k: v for k, v in CONFIG.items() if k != "class_weights"}, indent=2))
print(f"\nclass_weights[:10] = {CONFIG['class_weights'][:10]} ... (len={len(CONFIG['class_weights'])})")

In [ ]:
# ── Section 5: Training ────────────────────────────────────────────────────
t0 = time.time()

full_train_kwargs = dict(
    data=str(DATA_YAML_PATH),
    epochs=CONFIG["epochs"],
    patience=CONFIG["patience"],
    imgsz=CONFIG["img_size"],
    batch=CONFIG["batch_size"],
    optimizer=CONFIG["optimizer"],
    lr0=CONFIG["lr0"],
    lrf=CONFIG["lrf"],
    cos_lr=(CONFIG["scheduler"] == "cosine"),
    weight_decay=CONFIG["weight_decay"],
    warmup_epochs=CONFIG["warmup_epochs"],
    seed=CONFIG["seed"],
    device=CONFIG["device"],
    project="runs_nb3",
    name="yolo26n_sem",
    exist_ok=True,
    plots=True,
    cache="ram",
    workers=4,
    close_mosaic=5,
    amp=True,
)

# Minimal, near-universally-supported kwarg set — used only if the full set above fails
safe_train_kwargs = dict(
    data=str(DATA_YAML_PATH),
    epochs=CONFIG["epochs"],
    patience=CONFIG["patience"],
    imgsz=CONFIG["img_size"],
    batch=CONFIG["batch_size"],
    optimizer=CONFIG["optimizer"],
    lr0=CONFIG["lr0"],
    seed=CONFIG["seed"],
    device=CONFIG["device"],
    project="runs_nb3",
    name="yolo26n_sem",
    exist_ok=True,
    plots=True,
)

try:
    results = model.train(**full_train_kwargs)
    print("Trained with full kwarg set (cache/workers/close_mosaic/amp all accepted).")
except TypeError as e:
    print(f"WARNING: full training kwargs rejected ({e}). Retrying with minimal safe kwarg set...")
    model = YOLO(CHECKPOINT)  # reload fresh, since a failed .train() call can leave the model in a bad state
    results = model.train(**safe_train_kwargs)
    print("Trained with minimal safe kwarg set.")

total_train_time_sec = time.time() - t0
CONFIG["total_training_time_sec"] = round(total_train_time_sec, 1)
CONFIG["total_training_time_hms"] = time.strftime("%H:%M:%S", time.gmtime(total_train_time_sec))
print("Total training time:", CONFIG["total_training_time_hms"])

# ---- Plot per-epoch training loss & validation mIoU ----
run_dir = Path(model.trainer.save_dir)
res_csv = run_dir / "results.csv"
df_res = pd.read_csv(res_csv)
df_res.columns = [c.strip() for c in df_res.columns]
print(df_res.columns.tolist())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
loss_cols = [c for c in df_res.columns if "loss" in c.lower() and "train" in c.lower()]
for c in loss_cols:
    axes[0].plot(df_res["epoch"], df_res[c], label=c)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title("Training loss per epoch")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

miou_col = next((c for c in df_res.columns if "miou" in c.lower() or ("mask" in c.lower() and "map50-95" in c.lower())), None)
if miou_col:
    axes[1].plot(df_res["epoch"], df_res[miou_col], color="darkgreen", label=miou_col)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val mIoU / mask mAP")
axes[1].set_title("Validation mIoU per epoch"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / "nb3_train_curves.png", dpi=150)
plt.show()

In [ ]:
# ── Section 6: Test evaluation (held-out test set only) ──────────────────
best_weights = run_dir / "weights" / "best.pt"
best_model = YOLO(str(best_weights))

# 6a. Ultralytics' own semantic val metrics (best-effort, won't crash if attrs differ)
try:
    ul_test_metrics = best_model.val(data=str(DATA_YAML_PATH), split="test", imgsz=CONFIG["img_size"])
    print("Ultralytics built-in mIoU (model.val().miou):", getattr(ul_test_metrics, "miou", "N/A"))
    print("Ultralytics results_dict:", getattr(ul_test_metrics, "results_dict", "N/A"))
except Exception as e:
    print(f"WARNING: model.val() failed ({e}). Continuing with custom pixel-level eval only.")

# ---- Diagnose class indexing (best-effort; falls back to identity mapping if anything is off) ----
model_names = dict(best_model.names)
print("\nModel class names (from trained checkpoint):", model_names)

name_to_nb0_id = {name: (i + 1) for i, name in enumerate(CLASS_NAMES)}
pred_idx_to_nb0_id = {}
matched = 0
for model_idx, name in model_names.items():
    if name in name_to_nb0_id:
        pred_idx_to_nb0_id[model_idx] = name_to_nb0_id[name]
        matched += 1
    else:
        pred_idx_to_nb0_id[model_idx] = 0  # unmatched name (e.g. auto "background") -> bg

print(f"Matched {matched}/{NUM_CLASSES_FG} foreground class names.")
if matched < NUM_CLASSES_FG * 0.5:
    print("WARNING: fewer than half the class names matched — falling back to IDENTITY remap "
          "(model_idx -> model_idx+1, i.e. assume no explicit bg class from the model). "
          "Metrics below may need re-checking once you're back at the machine.")
    pred_idx_to_nb0_id = {k: min(k + 1, NUM_CLASSES_FG) for k in model_names.keys()}

print("Prediction-index -> NB0-class-id remap:", pred_idx_to_nb0_id)
max_model_idx = max(pred_idx_to_nb0_id.keys())
remap_lut = np.zeros(max_model_idx + 1, dtype=np.uint8)
for k, v in pred_idx_to_nb0_id.items():
    remap_lut[k] = v

_printed_debug_once = {"done": False}

def get_semantic_pred_mask(pred_result, h, w):
    """Extract dense per-pixel class-ID map from a Results object, trying multiple
    possible attribute layouts, and remap to NB0 convention. Never raises — returns
    an all-background mask as last resort so one bad image can't kill the whole run."""
    try:
        # Path 1: true semantic segmentation output (expected for yolo26*-sem models)
        if hasattr(pred_result, "semantic_mask") and pred_result.semantic_mask is not None:
            raw = pred_result.semantic_mask.data.cpu().numpy().astype(np.int64)
            if raw.shape != (h, w):
                raw = cv2.resize(raw.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(np.int64)
            raw = np.clip(raw, 0, max_model_idx)
            return remap_lut[raw]

        # Path 2: fallback — instance-style masks + boxes (in case checkpoint behaves like -seg)
        if getattr(pred_result, "masks", None) is not None and getattr(pred_result, "boxes", None) is not None:
            sem = np.zeros((h, w), dtype=np.uint8)
            masks = pred_result.masks.data.cpu().numpy()
            cls_ids = pred_result.boxes.cls.cpu().numpy().astype(int)
            confs = pred_result.boxes.conf.cpu().numpy()
            order = np.argsort(confs)
            for i in order:
                m = cv2.resize(masks[i], (w, h), interpolation=cv2.INTER_NEAREST) > 0.5
                remapped_id = pred_idx_to_nb0_id.get(cls_ids[i], 0)
                sem[m] = remapped_id
            return sem

        # Path 3: nothing usable found
        if not _printed_debug_once["done"]:
            print(f"WARNING: unrecognized Results object layout. Available attrs: {dir(pred_result)}")
            _printed_debug_once["done"] = True
        return np.zeros((h, w), dtype=np.uint8)

    except Exception as e:
        if not _printed_debug_once["done"]:
            print(f"WARNING: prediction-mask extraction failed ({e}). Using blank mask for this image and continuing.")
            _printed_debug_once["done"] = True
        return np.zeros((h, w), dtype=np.uint8)

# 6b. Pixel-level semantic metrics (mIoU, per-class IoU, OA, mean pixel acc, Dice, confusion matrix)
test_pairs = [resolve_img_label(p) for p in split_manifest["test"]]

all_gt, all_pred = [], []
per_image_iou = []
skipped_images = []

for img_path, lbl_path in test_pairs:
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            skipped_images.append(str(img_path))
            continue

        h, w = img.shape[:2]
        gt_mask = sc.polygons_to_mask(str(lbl_path), h, w, bg_is_zero=True)

        pred_result = best_model.predict(str(img_path), imgsz=CONFIG["img_size"], verbose=False)[0]
        pred_mask = get_semantic_pred_mask(pred_result, h, w)

        all_gt.append(gt_mask.flatten())
        all_pred.append(pred_mask.flatten())

        inter = np.logical_and(gt_mask > 0, pred_mask == gt_mask).sum()
        union = np.logical_or(gt_mask > 0, pred_mask > 0).sum()
        per_image_iou.append(inter / union if union > 0 else np.nan)
    except Exception as e:
        print(f"WARNING: unexpected error on {img_path} ({e}), skipping this image.")
        skipped_images.append(str(img_path))
        continue

print(f"\nTotal skipped/unreadable test images: {len(skipped_images)} out of {len(test_pairs)}")
assert all_gt, "No test images could be evaluated at all — check DATA_ROOT / symlinks."

y_true = np.concatenate(all_gt)
y_pred = np.concatenate(all_pred)
labels_all = list(range(NUM_CLASSES))

cm = confusion_matrix(y_true, y_pred, labels=labels_all)
tp = np.diag(cm).astype(np.float64)
fp = cm.sum(axis=0) - tp
fn = cm.sum(axis=1) - tp
iou_per_class = tp / (tp + fp + fn + 1e-9)
dice_per_class = 2 * tp / (2 * tp + fp + fn + 1e-9)

class_labels = ["background"] + [CLASS_NAMES[i] for i in range(NUM_CLASSES_FG)]
df_class_metrics = pd.DataFrame({
    "class_id": labels_all,
    "class_name": class_labels,
    "IoU": iou_per_class,
    "Dice": dice_per_class,
    "support_px": cm.sum(axis=1),
})

mIoU = iou_per_class[1:].mean()
mDice = dice_per_class[1:].mean()
overall_pixel_acc = tp.sum() / cm.sum()
mean_pixel_acc = (tp / (cm.sum(axis=1) + 1e-9))[1:].mean()

print(f"\nmIoU (fg only):            {mIoU:.4f}")
print(f"Mean Dice (fg only):       {mDice:.4f}")
print(f"Overall Pixel Accuracy:    {overall_pixel_acc:.4f}")
print(f"Mean Pixel Accuracy (fg):  {mean_pixel_acc:.4f}")
df_class_metrics.sort_values("IoU", ascending=False).to_csv(WORK_DIR / "nb3_per_class_iou.csv", index=False)
df_class_metrics.sort_values("IoU").head(15)

In [ ]:
top20_ids = df_class_metrics.sort_values("support_px", ascending=False)["class_id"].head(20).tolist()
idx = [labels_all.index(i) for i in top20_ids]
cm_top = cm[np.ix_(idx, idx)].astype(float)
cm_top_norm = cm_top / (cm_top.sum(axis=1, keepdims=True) + 1e-9)

plt.figure(figsize=(10, 8))
plt.imshow(cm_top_norm, cmap="viridis")
plt.colorbar(label="row-normalized fraction")
tick_labels = [class_labels[i] for i in idx]
plt.xticks(range(len(idx)), tick_labels, rotation=90, fontsize=7)
plt.yticks(range(len(idx)), tick_labels, fontsize=7)
plt.title("Pixel-level confusion matrix — top 20 classes by support")
plt.xlabel("Predicted"); plt.ylabel("Ground truth")
plt.tight_layout()
plt.savefig(WORK_DIR / "nb3_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ── Section 7: Error analysis ─────────────────────────────────────────────
df_worst = pd.DataFrame({
    "img_path": [str(p) for p, _ in test_pairs if str(p) not in skipped_images],
    "per_image_iou": per_image_iou,
}).sort_values("per_image_iou")

print("Worst 10 test images by IoU:")
print(df_worst.head(10).to_string(index=False))

N_WORST = 6
fig, axes = plt.subplots(3, N_WORST, figsize=(3.2 * N_WORST, 9))
for i, row in enumerate(df_worst.head(N_WORST).itertuples()):
    try:
        img_path = Path(row.img_path)
        lbl_path = img_path.parents[1] / "labels" / (img_path.stem + ".txt")
        img = cv2.imread(str(img_path))
        if img is None:
            raise ValueError("image unreadable")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        gt_mask = sc.polygons_to_mask(str(lbl_path), h, w, bg_is_zero=True)
        pred_result = best_model.predict(str(img_path), imgsz=CONFIG["img_size"], verbose=False)[0]
        pred_mask = get_semantic_pred_mask(pred_result, h, w)

        axes[0, i].imshow(img); axes[0, i].set_title(f"IoU={row.per_image_iou:.2f}", fontsize=9); axes[0, i].axis("off")
        axes[1, i].imshow(gt_mask, cmap="tab20", vmin=0, vmax=NUM_CLASSES); axes[1, i].set_title("GT"); axes[1, i].axis("off")
        axes[2, i].imshow(pred_mask, cmap="tab20", vmin=0, vmax=NUM_CLASSES); axes[2, i].set_title("Pred"); axes[2, i].axis("off")
    except Exception as e:
        print(f"WARNING: failed to visualize {row.img_path} ({e}), leaving panel blank.")
        axes[0, i].axis("off"); axes[1, i].axis("off"); axes[2, i].axis("off")

plt.suptitle("Task G — Worst test images: image / ground truth / prediction")
plt.tight_layout()
plt.savefig(WORK_DIR / "nb3_worst_images.png", dpi=150)
plt.show()

cm_fg = cm[1:, 1:].astype(float)
np.fill_diagonal(cm_fg, 0)
pairs = []
for i in range(cm_fg.shape[0]):
    for j in range(cm_fg.shape[1]):
        if cm_fg[i, j] > 0:
            pairs.append((CLASS_NAMES[i], CLASS_NAMES[j], cm_fg[i, j]))
df_confused = pd.DataFrame(pairs, columns=["true_class", "pred_class", "pixels"]).sort_values("pixels", ascending=False)
print("\nTop 15 most confused class pairs:")
print(df_confused.head(15).to_string(index=False))
print(
    "\nLikely causes: visually similar cooked foods (color/texture overlap), "
    "severe long-tail imbalance from NB0 EDA, small garnish/side-item objects, "
    "and boundary ambiguity where sauces/liquids blend into adjacent food regions."
)

In [ ]:
# ── Section 8 (NB3-local): Save this run's test summary for later cross-notebook comparison ──
yolo_test_summary = {
    "model": CONFIG["variant"],
    "checkpoint": CONFIG["model_checkpoint"],
    "params_M": CONFIG["params_M"],
    "pretrained_cityscapes_mIoU": CONFIG["pretrained_cityscapes_mIoU"],
    "img_size": CONFIG["img_size"],
    "batch_size": CONFIG["batch_size"],
    "optimizer": CONFIG["optimizer"],
    "lr0": CONFIG["lr0"],
    "epochs_trained": int(df_res["epoch"].max()) + 1,
    "total_training_time_sec": CONFIG["total_training_time_sec"],
    "total_training_time_hms": CONFIG["total_training_time_hms"],
    "test_mIoU": float(mIoU),
    "test_mean_dice": float(mDice),
    "test_overall_pixel_acc": float(overall_pixel_acc),
    "test_mean_pixel_acc": float(mean_pixel_acc),
    "test_images_evaluated": len(per_image_iou),
    "test_images_skipped": len(skipped_images),
}

summary_path = WORK_DIR / "yolo26n_test_summary.json"
with open(summary_path, "w") as f:
    json.dump(yolo_test_summary, f, indent=2)

print(f"Saved -> {summary_path}")
print(json.dumps(yolo_test_summary, indent=2))